# Learning Parameters from Arbitrary Data

This notebook shows how to use BFF when the training data were produced outside
the built-in molecular-dynamics workflow. The committed `.dat` files mimic a
water-model calibration campaign: each simulation row contains two sampled
Lennard-Jones oxygen parameters and three resulting liquid properties.

The data are realistic synthetic values. Replace the text files and mappings
below with your own parameter samples, simulated observables, and targets.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from bff.bayes.learning import LearningProblem, fit_surrogates
from bff.domain.specs import ChargeConstraint, Specs
from bff.io.logs import Logger
from bff.plotting import plot_corner, plot_marginals
from bff.qoi.data import QoIDataset

ROOT = Path.cwd()
if not (ROOT / 'raw-data').exists():
    ROOT = Path('examples/arbitrary-data').resolve()
RAW_DIR = ROOT / 'raw-data'
GENERATED_DIR = ROOT / 'generated'
MODEL_DIR = GENERATED_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

_ = torch.manual_seed(7)

## 1. Load the external tables

`simulation-results.dat` contains arbitrary training samples evaluated by an
external simulator. `experimental-targets.dat` contains one reference value
and uncertainty per observable. Both are whitespace-delimited numeric tables
with commented headers, so they can be loaded directly with `np.loadtxt`.

In [ ]:
simulation_columns = (
    'epsilon_O_kJ_mol',
    'sigma_O_nm',
    'density_kg_m3',
    'enthalpy_vap_kJ_mol',
    'diffusion_1e9_m2_s',
)
simulation_values = np.loadtxt(RAW_DIR / 'simulation-results.dat')
simulation_table = dict(zip(simulation_columns, simulation_values.T))

target_columns = ('density', 'enthalpy_vap', 'diffusion')
target_values, target_uncertainties = np.loadtxt(
    RAW_DIR / 'experimental-targets.dat'
)
targets = dict(zip(target_columns, target_values))
uncertainties = dict(zip(target_columns, target_uncertainties))

print(f'{len(simulation_values)} external simulation rows')
print(simulation_columns)
print('Targets:', targets)
print('Uncertainties:', uncertainties)

## 2. Convert the tables into BFF datasets

A BFF `QoIDataset` holds a common parameter matrix, simulated outputs, and the
matching reference output vector for one quantity of interest. Scalar and
vector-valued observables use the same object.

`Specs` provides the stable parameter order and the bounds used during
posterior learning. This example has no charge equations, so
`charge_constraints` is empty.

In [ ]:
specs = Specs({
    'bounds': {
        'epsilon O': [0.58, 0.718],
        'sigma O': [0.25, 0.38],
    },
    'charge_constraints': [],
})
parameter_columns = {
    'epsilon O': 'epsilon_O_kJ_mol',
    'sigma O': 'sigma_O_nm',
}
observable_columns = {
    'density': ('density_kg_m3', 'kg m^-3'),
    'enthalpy_vap': ('enthalpy_vap_kJ_mol', 'kJ mol^-1'),
    'diffusion': ('diffusion_1e9_m2_s', '10^-9 m^2 s^-1'),
}

parameter_names = specs.parameter_names(explicit_only=True)
inputs = np.column_stack([
    simulation_table[parameter_columns[name]] for name in parameter_names
])
datasets = {}
for name, (column, unit) in observable_columns.items():
    dataset = QoIDataset(
        name=name,
        inputs=inputs,
        outputs=np.asarray(simulation_table[column]).reshape(-1, 1),
        outputs_ref=np.array([targets[name]]),
        labels=('water at 298 K',),
        nuisance=float(uncertainties[name]),
        metadata={'unit': unit, 'source': 'external DAT'},
    )
    dataset.write(GENERATED_DIR / f'qoi-{name}.pt')
    datasets[name] = dataset

print('Parameter order:', parameter_names)
for dataset in datasets.values():
    print(dataset)

## 3. Fit local Gaussian-process models

BFF fits one surrogate per observable on CUDA and uses the
mean simulated value as the constant mean function for each surrogate.

In [ ]:
fit_logger = Logger('fit', GENERATED_DIR / 'fit.log', mode='w')
models = fit_surrogates(
    list(datasets.values()),
    y_means={
        name: float(dataset.outputs.mean())
        for name, dataset in datasets.items()
    },
    model_paths={
        name: MODEL_DIR / f'{name}.lgp' for name in datasets
    },
    reuse_models=False,
    n_hyper_max=36,
    committee_size=1,
    test_fraction=0.2,
    device='cuda',
    logger=fit_logger,
    max_iter=2000,
)

{name: f'{model.error:.3f}% SMAPE' for name, model in models.items()}

## 4. Learn the parameters

`LearningProblem` combines the fitted surrogates with the parameter bounds.
The target values and uncertainties are already stored in the models through
the datasets. Posterior sampling therefore needs only the models and specs.

In [ ]:
problem = LearningProblem.from_models(models, constraint=ChargeConstraint(specs))
learn_logger = Logger('learn', GENERATED_DIR / 'learn.log', mode='w')
results = problem.learn(
    priors_disttype='uniform',
    total_steps=1200,
    warmup=300,
    progress_stride=300,
    n_walkers=16,
    fn_checkpoint=GENERATED_DIR / 'mcmc-checkpoint.pt',
    fn_posterior=GENERATED_DIR / 'posterior.pt',
    fn_priors=GENERATED_DIR / 'priors.pt',
    restart=False,
    device='cuda',
    logger=learn_logger,
    rhat_tol=1.05,
    ess_min=100,
)
results.prepare_samples(discard=0, thin=1, strip_outliers=False)
results.write_summary(GENERATED_DIR / 'posterior-summary.yaml')
results.summary()

## 5. Visualize the posterior

The posterior should concentrate near `epsilon O = 0.65 kJ mol^-1` and
`sigma O = 0.315 nm`, the values used to construct the synthetic targets.

In [ ]:
plot_corner(results, fn_out=GENERATED_DIR / 'corner.png')
plot_marginals(results, specs, fn_out=GENERATED_DIR / 'marginals.png')